# AI Panel Data — Exploration Notebook

This notebook loads the merged panel and demonstrates common operations:
descriptive statistics, coverage heatmaps, cross-sectional plots, and
a simple OLS regression.

**Run the pipeline first:**
```bash
python main.py
```

In [ ]:
import sys
sys.path.insert(0, '..')   # make src/ importable from notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Nicer default plots
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('tab10')

PANEL_PATH = '../data/processed/panel.csv'
CODEBOOK_PATH = '../data/processed/codebook.csv'

## 1. Load & inspect

In [ ]:
df = pd.read_csv(PANEL_PATH)
codebook = pd.read_csv(CODEBOOK_PATH)

print(f'Shape : {df.shape}')
print(f'Countries : {df.iso3.nunique()}')
print(f'Years     : {df.year.min()} – {df.year.max()}')
df.head()

In [ ]:
# Codebook: coverage overview
codebook.sort_values('pct_non_missing', ascending=False)

## 2. Data coverage heatmap

In [ ]:
# Show % non-missing per indicator (top 30 most complete)
top_cols = codebook.sort_values('pct_non_missing', ascending=False).head(30)['column'].tolist()
top_cols = [c for c in top_cols if c not in ('iso3', 'year', 'country_name', 'oecd_member')]

coverage = df[top_cols].notna().mean() * 100
fig, ax = plt.subplots(figsize=(10, 6))
coverage.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% non-missing observations')
ax.set_title('Data coverage by indicator')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

## 3. R&D spending over time (OECD countries)

In [ ]:
# Highlight a selection of countries
highlight = ['USA', 'CHN', 'KOR', 'DEU', 'JPN', 'GBR', 'SWE', 'ISR']

if 'rd_total_pct_gdp' in df.columns:
    fig, ax = plt.subplots(figsize=(11, 5))
    for iso3 in highlight:
        sub = df[df['iso3'] == iso3].dropna(subset=['rd_total_pct_gdp'])
        ax.plot(sub['year'], sub['rd_total_pct_gdp'], marker='o', markersize=3, label=iso3)
    ax.set_xlabel('Year')
    ax.set_ylabel('R&D expenditure (% of GDP)')
    ax.set_title('Total R&D spending as % of GDP')
    ax.legend(loc='upper left', ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('rd_total_pct_gdp column not present — check OECD fetch.')

## 4. AI patents: cross-section (latest year)

In [ ]:
if 'ai_patents_total' in df.columns:
    latest_year = df['year'].max()
    top20 = (
        df[df['year'] == latest_year]
        .dropna(subset=['ai_patents_total'])
        .nlargest(20, 'ai_patents_total')
    )
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top20['iso3'], top20['ai_patents_total'], color='coral')
    ax.set_xlabel('AI patent applications (PCT)')
    ax.set_title(f'Top 20 countries by AI patent applications ({latest_year})')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('ai_patents_total not present — check OECD patents fetch.')

## 5. Democracy × R&D scatter

In [ ]:
if 'vdem_liberal_democracy' in df.columns and 'rd_total_pct_gdp' in df.columns:
    latest_year = df['year'].max()
    scatter = df[df['year'] == latest_year].dropna(subset=['vdem_liberal_democracy', 'rd_total_pct_gdp'])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        scatter['vdem_liberal_democracy'],
        scatter['rd_total_pct_gdp'],
        alpha=0.7, s=40, color='steelblue'
    )
    for _, row in scatter.nlargest(10, 'rd_total_pct_gdp').iterrows():
        ax.annotate(row['iso3'], (row['vdem_liberal_democracy'], row['rd_total_pct_gdp']),
                    fontsize=7, ha='left')

    # OLS trend line
    x = scatter['vdem_liberal_democracy']
    y = scatter['rd_total_pct_gdp']
    m, b = np.polyfit(x.dropna(), y[x.notna()], 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m * xline + b, 'r--', linewidth=1)

    ax.set_xlabel('V-Dem Liberal Democracy Index')
    ax.set_ylabel('R&D as % of GDP')
    ax.set_title(f'Democracy vs R&D spending ({latest_year})')
    plt.tight_layout()
    plt.show()
else:
    print('V-Dem or R&D columns not present.')

## 6. Export a filtered panel (e.g. OECD members only)

In [ ]:
oecd_panel = df[df['oecd_member'] == 1].copy()
print(f'OECD panel: {len(oecd_panel)} rows, {oecd_panel.iso3.nunique()} countries')
oecd_panel.to_csv('../data/processed/panel_oecd_only.csv', index=False)
print('Saved → data/processed/panel_oecd_only.csv')

## 7. Quick OLS with statsmodels

Just to verify the panel works as an analysis input.
Install statsmodels if needed: `pip install statsmodels`

In [ ]:
try:
    import statsmodels.formula.api as smf

    reg_df = df.dropna(subset=['rd_total_pct_gdp', 'log_gdp_per_capita_const2015usd',
                                'vdem_liberal_democracy', 'internet_users_pct_pop'])

    model = smf.ols(
        'rd_total_pct_gdp ~ log_gdp_per_capita_const2015usd + vdem_liberal_democracy + internet_users_pct_pop + C(year)',
        data=reg_df
    ).fit(cov_type='HC3')   # heteroskedasticity-robust SEs

    print(model.summary())

except ImportError:
    print('statsmodels not installed. Run: pip install statsmodels')
except Exception as e:
    print(f'Regression failed (likely missing columns): {e}')